In [3]:
import os
import time
from pathlib import Path
from collections import Counter

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

In [4]:
PROJECT_ROOT = Path(
    r"D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics"
).resolve()
CLEANED_DIR = Path(os.environ.get("CLEANED_DIR", PROJECT_ROOT / "data" / "cleaned"))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "output"))
CHECKPOINT_PATH = str(Path(os.environ.get("CHECKPOINT_DIR", OUTPUT_DIR / "schedule_checkpoint")))

FINAL_PARQUET_PATH = str(OUTPUT_DIR / "final_schedule.parquet")
FINAL_CSV_DIR = str(OUTPUT_DIR / "final_schedule_csv")
FINAL_CSV_PATH = str(OUTPUT_DIR / "final_schedule.csv")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MYSQL_JAR = os.environ.get("MYSQL_JDBC_JAR", str(PROJECT_ROOT / "jdbc" / "mysql-connector-j-9.7.0.jar"))

N_CORES = int(os.environ.get("SPARK_CORES", "8"))

EXPECTED_FINAL_ROWS = os.environ.get("EXPECTED_FINAL_ROWS")
EXPECTED_FINAL_ROWS = int(EXPECTED_FINAL_ROWS) if EXPECTED_FINAL_ROWS else None

print(f"Cleaned dir:     {CLEANED_DIR}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_PATH}")
print(f"MySQL JDBC jar:  {MYSQL_JAR}")

Cleaned dir:     D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\data\cleaned
Output dir:      D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output
Checkpoint dir:  D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\schedule_checkpoint
MySQL JDBC jar:  D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\jdbc\mysql-connector-j-9.7.0.jar


### MySQL & Spark configuration

In [5]:
MYSQL_HOST = os.environ.get("MYSQL_HOST")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT"))
MYSQL_USER = os.environ.get("MYSQL_USER")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD")
MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE")

assert MYSQL_PASSWORD, "MYSQL_PASSWORD is not set."

MYSQL_URI = (
    f"mysql+pymysql://{MYSQL_USER}:{quote_plus(MYSQL_PASSWORD)}@"
    f"{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

MYSQL_JDBC_URL = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"

MYSQL_PROPERTIES = {
    "user": MYSQL_USER,
    "password": MYSQL_PASSWORD,
    "driver": "com.mysql.cj.jdbc.Driver",
}

print(f"MySQL target: {MYSQL_USER}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")

MySQL target: root@localhost:3306/bus_performance_analytics


In [4]:
spark = (
    SparkSession.builder
    .appName("BusRoute_02_DataStorageProcessing")
    .master(f"local[{N_CORES}]")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 2))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.autoBroadcastJoinThreshold", -1)
    .config("spark.jars", MYSQL_JAR)
    .config("spark.driver.extraClassPath", MYSQL_JAR)
    .config("spark.executor.extraClassPath", MYSQL_JAR)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("="*60)
print("Spark Version :", spark.version)
print("Parallelism   :", spark.sparkContext.defaultParallelism)
print("Shuffle Parts :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI      :", spark.sparkContext.uiWebUrl)
print("="*60)

Spark Version : 3.5.8
Parallelism   : 8
Shuffle Parts : 16
Spark UI      : http://DESKTOP-P7PE4MO:4040


## Load cleaned data

In [5]:
dataset_files = {
    "stop_times":        "timetable_stop_times.parquet",
    "vehicle_journeys":  "timetable_vehicle_journeys.parquet",
    "stops":             "timetable_stops.parquet",
    "fares":             "fares.parquet",
    "location":          "location_pings.parquet",
    "disruptions":       "disruptions.parquet",
}

raw_tables = {
    name: spark.read.parquet(str(CLEANED_DIR / filename))
    for name, filename in dataset_files.items()
}

stop_times = raw_tables["stop_times"]
vehicle_journeys = raw_tables["vehicle_journeys"]
stops = raw_tables["stops"]
fares = raw_tables["fares"]
location = raw_tables["location"]
disruptions = raw_tables["disruptions"]

print("="*70)
print(f"{'Dataset':<25}{'Rows':>12}{'Columns':>10}")
print("="*70)
for name, df in raw_tables.items():
    print(f"{name:<25}{df.count():>12,}{len(df.columns):>10}")

Dataset                          Rows   Columns
stop_times                    926,481         7
vehicle_journeys               35,997         9
stops                           8,895         3
fares                             126         9
location                       10,527        23
disruptions                       450        13


In [6]:
BASE_ROW_COUNT = stop_times.count()
print("Base dataset rows (timetable_stop_times):", f"{BASE_ROW_COUNT:,}")

print("\nDuplicate vehicle_journey_code (should be empty):")
vehicle_journeys.groupBy("source_file", "vehicle_journey_code") \
    .count().filter(F.col("count") > 1).show()

print("Duplicate stop_point_ref (should be empty):")
stops.groupBy("source_file", "stop_point_ref") \
    .count().filter(F.col("count") > 1).show()

Base dataset rows (timetable_stop_times): 926,481

Duplicate vehicle_journey_code (should be empty):
+-----------+--------------------+-----+
|source_file|vehicle_journey_code|count|
+-----------+--------------------+-----+
+-----------+--------------------+-----+

Duplicate stop_point_ref (should be empty):
+-----------+--------------+-----+
|source_file|stop_point_ref|count|
+-----------+--------------+-----+
+-----------+--------------+-----+



## Build `final_schedule` with validated broadcast joins

### Repartition the base table on the join key

In [7]:
schedule = stop_times.repartition(N_CORES * 4, "source_file", "vehicle_journey_code")
print("Base rows :", schedule.count())
print("Partitions:", schedule.rdd.getNumPartitions())
assert schedule.count() == BASE_ROW_COUNT

Base rows : 926481
Partitions: 32


### Join vehicle-journey metadata (operator, line, service, departure time)

In [8]:
vehicle_lookup = vehicle_journeys.select(
    "source_file",
    "vehicle_journey_code",
    "service_ref",
    "line_name",
    "operator_ref",
    "journey_pattern_ref",
    "scheduled_departure_time",
    "scheduled_departure_ts",
)

schedule = schedule.join(
    F.broadcast(vehicle_lookup),
    on=["source_file", "vehicle_journey_code"],
    how="left",
)

rows = schedule.count()
matched = schedule.filter(F.col("operator_ref").isNotNull()).count()
print("Rows    :", f"{rows:,}")
print("Matched :", f"{matched:,}", f"({100*matched/rows:.1f}%)")
assert rows == BASE_ROW_COUNT

Rows    : 926,481
Matched : 926,481 (100.0%)


### Join stop metadata (stop name)

In [9]:
stop_lookup = stops.select(
    "source_file",
    "stop_point_ref",
    F.col("common_name").alias("stop_name"),
)

schedule = schedule.join(
    F.broadcast(stop_lookup),
    on=["source_file", "stop_point_ref"],
    how="left",
)

rows = schedule.count()
matched = schedule.filter(F.col("stop_name").isNotNull()).count()
print("Rows          :", f"{rows:,}")
print("Matched stops :", f"{matched:,}", f"({100*matched/rows:.1f}%)")
assert rows == BASE_ROW_COUNT

Rows          : 926,481
Matched stops : 511,501 (55.2%)


### Join fare availability (line-level aggregate, joined once)

In [10]:
fare_lookup = (
    fares.groupBy("line_ref")
    .agg(F.count("*").alias("fare_publication_count"))
    .withColumn("has_fare_data", F.lit(1))
)
fare_lookup.show(5, truncate=False)

schedule = schedule.join(F.broadcast(fare_lookup), on="line_ref", how="left")
schedule = schedule.fillna({"fare_publication_count": 0, "has_fare_data": 0})

rows = schedule.count()
fare_rows = schedule.filter(F.col("has_fare_data") == 1).count()
print("Rows          :", f"{rows:,}")
print("Rows w/ fare  :", f"{fare_rows:,}", f"({100*fare_rows/rows:.1f}%)")
assert rows == BASE_ROW_COUNT

+---------------------+----------------------+-------------+
|line_ref             |fare_publication_count|has_fare_data|
+---------------------+----------------------+-------------+
|SCOX:PH0005863:15:9  |1                     |1            |
|SCGL:PH0005031:349:33|1                     |1            |
|SCGL:PH0005031:386:80|1                     |1            |
|SCOX:PH0005863:19:S5 |1                     |1            |
|SCOX:PH0005863:62:26 |1                     |1            |
+---------------------+----------------------+-------------+
only showing top 5 rows

Rows          : 926,481
Rows w/ fare  : 678,343 (73.2%)


In [11]:
dupe_cols = [c for c, n in Counter(schedule.columns).items() if n > 1]
assert not dupe_cols, f"Duplicate column names after joins: {dupe_cols}"

print("="*60)
print("Rows    :", f"{schedule.count():,}")
print("Columns :", len(schedule.columns))
print("="*60)
schedule.printSchema()

Rows    : 926,481
Columns : 16
root
 |-- line_ref: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- scheduled_time: string (nullable = true)
 |-- scheduled_ts: timestamp (nullable = true)
 |-- service_ref: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)
 |-- scheduled_departure_ts: timestamp (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- fare_publication_count: long (nullable = false)
 |-- has_fare_data: integer (nullable = false)



## Performance: cache, repartition, checkpoint

In [12]:
schedule = schedule.cache()
BASE_ROW_COUNT = schedule.count()  # re-anchor after the join chain above
print("Cached row count:", f"{BASE_ROW_COUNT:,}")
print("Partitions before repartition:", schedule.rdd.getNumPartitions())

Cached row count: 926,481
Partitions before repartition: 32


In [13]:
schedule = schedule.repartition(N_CORES)
print("Partitions after repartition:", schedule.rdd.getNumPartitions())

rows = schedule.count()
print("Rows:", f"{rows:,}")
assert rows == BASE_ROW_COUNT

Partitions after repartition: 8
Rows: 926,481


In [14]:
spark.sparkContext.setCheckpointDir(CHECKPOINT_PATH)
schedule = schedule.checkpoint(eager=True)
print(f"Checkpoint written to {CHECKPOINT_PATH}")

Checkpoint written to D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\schedule_checkpoint


In [15]:
schedule.explain(mode="formatted")

== Physical Plan ==
* Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [16]: [line_ref#2, source_file#0, stop_point_ref#3, vehicle_journey_code#1, stop_sequence#4, scheduled_time#5, scheduled_ts#6, service_ref#16, line_name#18, operator_ref#19, journey_pattern_ref#20, scheduled_departure_time#21, scheduled_departure_ts#22, stop_name#370, fare_publication_count#491L, has_fare_data#492]
Arguments: [line_ref#2, source_file#0, stop_point_ref#3, vehicle_journey_code#1, stop_sequence#4, scheduled_time#5, scheduled_ts#6, service_ref#16, line_name#18, operator_ref#19, journey_pattern_ref#20, scheduled_departure_time#21, scheduled_departure_ts#22, stop_name#370, fare_publication_count#491L, has_fare_data#492], MapPartitionsRDD[292] at checkpoint at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)




In [16]:
natural_key = ["source_file", "vehicle_journey_code", "stop_point_ref", "stop_sequence"]
n_total = schedule.count()
n_distinct_key = schedule.dropDuplicates(natural_key).count()
print("Total rows          :", f"{n_total:,}")
print("Distinct on nat. key:", f"{n_distinct_key:,}")
assert n_total == n_distinct_key, "Unexpected duplicate rows on the natural key -- investigate the joins above."

if EXPECTED_FINAL_ROWS is not None:
    print(f"\nExpected final row count: {EXPECTED_FINAL_ROWS:,}")
    assert n_total == EXPECTED_FINAL_ROWS, (
        f"Final row count {n_total:,} does not match EXPECTED_FINAL_ROWS "
        f"{EXPECTED_FINAL_ROWS:,} -- confirm whether this is expected before proceeding."
    )
print("No unexpected duplicates found.")

Total rows          : 926,481
Distinct on nat. key: 926,481
No unexpected duplicates found.


## Persist `final_schedule`

In [17]:
schedule.write.mode("overwrite").parquet(FINAL_PARQUET_PATH)
print(f"Final Parquet saved to: {FINAL_PARQUET_PATH}")

Final Parquet saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\final_schedule.parquet


In [18]:
verify = spark.read.parquet(FINAL_PARQUET_PATH)
print("Rows    :", f"{verify.count():,}")
print("Columns :", len(verify.columns))
verify.printSchema()

Rows    : 926,481
Columns : 16
root
 |-- line_ref: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- scheduled_time: string (nullable = true)
 |-- scheduled_ts: timestamp (nullable = true)
 |-- service_ref: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)
 |-- scheduled_departure_ts: timestamp (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- fare_publication_count: long (nullable = true)
 |-- has_fare_data: integer (nullable = true)



In [19]:
import glob, shutil

(
    schedule.coalesce(1).write
    .mode("overwrite")
    .option("header", True)
    .csv(FINAL_CSV_DIR)
)
part_file = glob.glob(str(Path(FINAL_CSV_DIR) / "part-*.csv"))[0]
if Path(FINAL_CSV_PATH).exists():
    Path(FINAL_CSV_PATH).unlink()
shutil.move(part_file, FINAL_CSV_PATH)
shutil.rmtree(FINAL_CSV_DIR)
print(f"Final CSV saved to: {FINAL_CSV_PATH}")

Final CSV saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\final_schedule.csv


##  MySQL persistence

In [20]:
bootstrap_engine = create_engine(MYSQL_URI.rsplit("/", 1)[0] + "/")
with bootstrap_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}`"))
    conn.commit()
bootstrap_engine.dispose()
print(f"Database '{MYSQL_DATABASE}' is ready.")

Database 'bus_performance_analytics' is ready.


In [21]:
engine = create_engine(MYSQL_URI)

def fare_publication_count_for_operator(operator_ref: str) -> int:
    '''Example of a parameterised (bound-parameter) query: safe against SQL
    injection because the value is never string-concatenated into the SQL text.'''
    query = text('''
        SELECT COUNT(*) AS n
        FROM fares
        WHERE operator_ref = :operator_ref
    ''')
    with engine.connect() as conn:
        result = conn.execute(query, {"operator_ref": operator_ref}).fetchone()
    return result.n if result else 0

print("Connected. Parameterised query helper ready "
      "(will be exercised after the 'fares' table is written below).")

Connected. Parameterised query helper ready (will be exercised after the 'fares' table is written below).


In [22]:
def write_spark_df_to_mysql(df, table_name):
    start = time.time()
    temp_df = df.select(*list(dict.fromkeys(df.columns)))  # drop any duplicate columns defensively

    for field in temp_df.schema.fields:
        if field.dataType.typeName() == "timestamp":
            temp_df = temp_df.withColumn(
                field.name, F.date_format(F.col(field.name), "yyyy-MM-dd HH:mm:ss")
            )

    row_count = temp_df.count()
    (
        temp_df.write.mode("overwrite")
        .jdbc(url=MYSQL_JDBC_URL, table=table_name, properties=MYSQL_PROPERTIES)
    )
    elapsed = time.time() - start
    print(f"{table_name:<30}{row_count:>10,} rows{elapsed:>12.2f} sec")


tables = {
    "enriched_schedule": schedule,
    "timetable_vehicle_journeys": vehicle_journeys,
    "timetable_stops": stops,
    "fares": fares,
}

print("Writing tables...\n")
for table_name, dataframe in tables.items():
    write_spark_df_to_mysql(dataframe, table_name)
print("\nAll tables written successfully.")

Writing tables...

enriched_schedule                926,481 rows       86.36 sec
timetable_vehicle_journeys        35,997 rows        5.03 sec
timetable_stops                    8,895 rows        1.47 sec
fares                                126 rows        0.55 sec

All tables written successfully.


In [23]:
print("="*60)
print("MYSQL TABLE VERIFICATION")
print("="*60)
for table in tables:
    count = spark.read.jdbc(url=MYSQL_JDBC_URL, table=table, properties=MYSQL_PROPERTIES).count()
    print(f"{table:<30}{count:,} rows")

# Exercise the parameterised query helper from Cell 21 now that 'fares' exists.
sample_operator = fares.select("operator_ref").filter(F.col("operator_ref").isNotNull()).first()
if sample_operator:
    n = fare_publication_count_for_operator(sample_operator["operator_ref"])
    print(f"\nParameterised query check -- fare publications for operator "
          f"'{sample_operator['operator_ref']}': {n}")

MYSQL TABLE VERIFICATION
enriched_schedule             926,481 rows
timetable_vehicle_journeys    35,997 rows
timetable_stops               8,895 rows
fares                         126 rows

Parameterised query check -- fare publications for operator 'noc:SCGL': 84


In [24]:
mysql_schedule = spark.read.jdbc(url=MYSQL_JDBC_URL, table="enriched_schedule", properties=MYSQL_PROPERTIES)
mysql_schedule.createOrReplaceTempView("mysql_schedule")

spark.sql('''
    SELECT operator_ref, COUNT(*) AS total_trips
    FROM mysql_schedule
    GROUP BY operator_ref
    ORDER BY total_trips DESC
    LIMIT 10
''').show(truncate=False)

+------------+-----------+
|operator_ref|total_trips|
+------------+-----------+
|1           |598852     |
|11          |186878     |
|2           |65351      |
|4           |49557      |
|3           |25843      |
+------------+-----------+



In [25]:
print("="*70)
print("MYSQL PERSISTENCE SUMMARY")
print("="*70)
print(f"Database Name : {MYSQL_DATABASE}")
print(f"Tables Stored : {len(tables)}")
for table in tables:
    rows = spark.read.jdbc(url=MYSQL_JDBC_URL, table=table, properties=MYSQL_PROPERTIES).count()
    print(f"{table:<30}{rows:,} rows")
print("\nAll processed datasets have been persisted to Parquet, CSV, and MySQL.")

MYSQL PERSISTENCE SUMMARY
Database Name : bus_performance_analytics
Tables Stored : 4
enriched_schedule             926,481 rows
timetable_vehicle_journeys    35,997 rows
timetable_stops               8,895 rows
fares                         126 rows

All processed datasets have been persisted to Parquet, CSV, and MySQL.


In [26]:
spark.stop()
print("Spark session stopped. Notebook 02 complete.")

Spark session stopped. Notebook 02 complete.
